## 1. Chunks 切分

In [1]:
import os, re
import pandas as pd
import numpy as np

def get_chunks(file_name):
    data = pd.read_parquet(file_name)
    window_size = 10 * 250
    chunks = [data[i:i + window_size] for i in range(0, len(data), window_size)][:-1]
    return chunks

data_path = 'E:/ECG_data/250hz/records/raw/'
ecg_files = [file for file in os.listdir(data_path) if 'ecg_' in file]

ecg_dicts = {}
for file in ecg_files:
    time_str = re.search(r'\d{4}-\d{2}-\d{2}_\d{6}', file).group()
    chunks = get_chunks(data_path+file)
    ecg_dicts[time_str] = chunks

In [ ]:
chunks = ecg_dicts['2026-02-20_010744']
for chunk in chunks:
    chunk = chunk.copy()
    chunk['chunk_idx'] = chunk.index
    chunk = chunk.loc[:, ['time', 'ecg', 'chunk_idx']]
    break

In [6]:
chunks

[              time  ecg  lead  chunk_idx
 0     1.771521e+09  317     0          0
 1     1.771521e+09  319     0          1
 2     1.771521e+09  321     0          2
 3     1.771521e+09  321     0          3
 4     1.771521e+09  322     0          4
 ...            ...  ...   ...        ...
 2495  1.771521e+09  360     0       2495
 2496  1.771521e+09  331     0       2496
 2497  1.771521e+09  309     0       2497
 2498  1.771521e+09  306     0       2498
 2499  1.771521e+09  317     0       2499
 
 [2500 rows x 4 columns],
               time  ecg  lead  chunk_idx
 2500  1.771521e+09  328     0       2500
 2501  1.771521e+09  331     0       2501
 2502  1.771521e+09  328     0       2502
 2503  1.771521e+09  324     0       2503
 2504  1.771521e+09  333     0       2504
 ...            ...  ...   ...        ...
 4995  1.771521e+09  328     0       4995
 4996  1.771521e+09  329     0       4996
 4997  1.771521e+09  330     0       4997
 4998  1.771521e+09  329     0       4998
 4999 

## 2. 脚本划分 good 和 bad

In [4]:
from ecg_core.rr_hr_hrv import *

def nan_in_rrs(chunk):
    my_beats = BeatCalc(chunk)
    my_rr_s, my_rr_clean = my_beats.get_clean_rr()
    return my_rr_s.hasnans

def get_good_bad_labels(chunks):
    labels = np.zeros(len(chunks))
    for i, chunk in enumerate(chunks):
        if 1 in chunk['lead']:
            labels[i] = 1
        elif nan_in_rrs(chunk):
            labels[i] = 1
        elif len(chunk[chunk['ecg']>600]) > 10 or len(chunk[chunk['ecg']<100])>30:
            labels[i] = 1
    return labels


ecg_labels = {}
for key, chunks in ecg_dicts.items():
    if key == '2026-03-01_225443':
        labels = get_good_bad_labels(chunks)
        ecg_labels[key] = labels
        good_idx = np.where(labels == 0)[0]
        bad_idx = np.where(labels == 1)[0]
        print(key, f"Good: {len(good_idx)}", f"Bad: {len(bad_idx)}")

d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value enco

2026-03-01_225443 Good: 2742 Bad: 328


## 3. 保存标注 parquet

In [5]:
label_map = {0: "good", 1: "bad"}

for key, chunks in ecg_dicts.items():
    if key == '2026-03-01_225443':
        labels = ecg_labels[key]
        str_labels = [label_map[label] for label in labels]
        plabel_dict = {idx: [chunks[idx]['ecg'].values, str_labels[idx]] for idx in range(len(chunks))}
        plabel_df = pd.DataFrame(plabel_dict).T
        plabel_df.columns = ['ecg_raw', 'label']
        plabel_df.to_parquet(f"{data_path}gb_chunks_{key}.parquet")